# 04 - Caracterizacao de Elastomeros
> Curvas tensao-deformacao, modelos constitutivos e comparacao com literatura

**Modulo:** `13_enfitec_projetos` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print('Bibliotecas carregadas!')

## Gerando Dados de Ensaio de Tracao

Vamos gerar dados sinteticos que simulam o comportamento hiperelastico
tipico de um elastomero (borracha natural/NBR) sob tracao uniaxial.
O comportamento inclui:
- Regiao inicial quasi-linear (baixas deformacoes)
- Amolecimento intermediario (efeito Mullins)
- Enrijecimento por cristalizacao (altas deformacoes)


In [ ]:
np.random.seed(42)

# Deformacao de engenharia: 0 a 500%
strain = np.linspace(0, 5.0, 500)

# Lambda (razao de alongamento)
lam = 1 + strain

# Modelo hiperelastico simplificado (tipo Mooney-Rivlin + cristalizacao)
C1 = 0.18   # MPa
C2 = 0.02   # MPa
C3 = 0.0008 # Coeficiente de enrijecimento

# Tensao de engenharia (Mooney-Rivlin + termo de enrijecimento)
sigma_MR = 2 * (lam - 1/lam**2) * (C1 + C2 / lam)
sigma_stiffening = C3 * (lam - 1)**3
stress = sigma_MR + sigma_stiffening

# Adicionar ruido experimental
noise = np.random.normal(0, 0.02, len(strain))
stress_exp = stress + noise
stress_exp[0] = 0  # Garantir inicio em zero
stress_exp = np.maximum(stress_exp, 0)  # Sem tensao negativa

print(f'Deformacao maxima: {strain[-1]*100:.0f}%')
print(f'Tensao maxima: {stress_exp[-1]:.2f} MPa')
print(f'Pontos de dados: {len(strain)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva completa
axes[0].plot(strain * 100, stress_exp, 'b-', linewidth=1.5, label='Dados experimentais')
axes[0].plot(strain * 100, stress, 'r--', linewidth=1, alpha=0.7, label='Modelo teorico')
axes[0].set_xlabel('Deformacao (%)')
axes[0].set_ylabel('Tensao (MPa)')
axes[0].set_title('Curva Tensao-Deformacao - Elastomero')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zoom na regiao inicial (0-50%)
mask = strain <= 0.5
axes[1].plot(strain[mask] * 100, stress_exp[mask], 'b-', linewidth=1.5)
axes[1].set_xlabel('Deformacao (%)')
axes[1].set_ylabel('Tensao (MPa)')
axes[1].set_title('Regiao Inicial (0-50%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Identificando Propriedades Mecanicas

Vamos extrair propriedades-chave da curva e pedir ao Claude para
interpretar os resultados e estimar dureza Shore A.


In [ ]:
# Calcular propriedades mecanicas
# Modulo inicial (tangente em deformacao ~1%)
idx_1pct = np.argmin(np.abs(strain - 0.01))
idx_5pct = np.argmin(np.abs(strain - 0.05))
modulo_inicial = (stress_exp[idx_5pct] - stress_exp[idx_1pct]) / (strain[idx_5pct] - strain[idx_1pct])

# Modulo a 100% de deformacao (M100)
idx_100 = np.argmin(np.abs(strain - 1.0))
M100 = stress_exp[idx_100]

# Modulo a 300% de deformacao (M300)
idx_300 = np.argmin(np.abs(strain - 3.0))
M300 = stress_exp[idx_300]

# Tensao maxima (resistencia a tracao)
tensile_strength = np.max(stress_exp)
elongation_break = strain[np.argmax(stress_exp)] * 100

print(f'Modulo inicial (tangente): {modulo_inicial:.2f} MPa')
print(f'M100 (tensao a 100%): {M100:.2f} MPa')
print(f'M300 (tensao a 300%): {M300:.2f} MPa')
print(f'Resistencia a tracao: {tensile_strength:.2f} MPa')
print(f'Alongamento na ruptura: {elongation_break:.0f}%')

In [ ]:
props_prompt = f"""Sou engenheiro de materiais e realizei um ensaio de tracao uniaxial
em um elastomero. Os resultados foram:

- Modulo inicial (tangente 1-5%): {modulo_inicial:.2f} MPa
- M100 (tensao a 100% deformacao): {M100:.2f} MPa
- M300 (tensao a 300% deformacao): {M300:.2f} MPa
- Resistencia a tracao: {tensile_strength:.2f} MPa
- Alongamento na ruptura: {elongation_break:.0f}%
- Razao M300/M100: {M300/M100:.2f}

Por favor:
1. Classifique o tipo provavel de elastomero com base nesses valores
2. Estime a dureza Shore A provavel
3. Avalie se a razao M300/M100 indica boa dispersao de carga
4. Comente sobre a qualidade geral do material

Responda de forma tecnica e objetiva."""

resp_props = ask(props_prompt, model=SONNET, max_tokens=1500)
print(resp_props)

## Modelos Constitutivos

Vamos ajustar dois modelos classicos de hiperelasticidade:
- **Neo-Hookean**: modelo mais simples, 1 parametro (C1)
- **Mooney-Rivlin**: 2 parametros (C1, C2), mais preciso


In [ ]:
model_prompt = """Preciso ajustar modelos constitutivos hiperelasticos a dados
de tracao uniaxial de um elastomero. Explique brevemente:

1. A expressao da tensao de engenharia para o modelo Neo-Hookean
   em tracao uniaxial (em funcao de lambda = 1 + epsilon)
2. A expressao para o modelo Mooney-Rivlin (2 parametros)
3. Limitacoes de cada modelo para grandes deformacoes

Seja conciso e use notacao matematica simples."""

resp_models = ask(model_prompt, model=SONNET, max_tokens=1200)
print(resp_models)

In [ ]:
from scipy.optimize import curve_fit

# Modelo Neo-Hookean: sigma_eng = 2*C1*(lambda - 1/lambda^2)
def neo_hookean(lam, C1):
    return 2 * C1 * (lam - 1.0 / lam**2)

# Modelo Mooney-Rivlin: sigma_eng = 2*(lambda - 1/lambda^2)*(C1 + C2/lambda)
def mooney_rivlin(lam, C1, C2):
    return 2 * (lam - 1.0 / lam**2) * (C1 + C2 / lam)

# Dados para ajuste (excluir ponto zero para evitar singularidade)
lam_fit = lam[1:]
stress_fit = stress_exp[1:]

# Ajuste Neo-Hookean
popt_nh, pcov_nh = curve_fit(neo_hookean, lam_fit, stress_fit, p0=[0.2])
C1_nh = popt_nh[0]

# Ajuste Mooney-Rivlin
popt_mr, pcov_mr = curve_fit(mooney_rivlin, lam_fit, stress_fit, p0=[0.2, 0.05])
C1_mr, C2_mr = popt_mr

print('=== Parametros Ajustados ===')
print(f'Neo-Hookean:   C1 = {C1_nh:.4f} MPa')
print(f'Mooney-Rivlin: C1 = {C1_mr:.4f} MPa, C2 = {C2_mr:.4f} MPa')

# Calcular R-quadrado
ss_res_nh = np.sum((stress_fit - neo_hookean(lam_fit, *popt_nh))**2)
ss_res_mr = np.sum((stress_fit - mooney_rivlin(lam_fit, *popt_mr))**2)
ss_tot = np.sum((stress_fit - np.mean(stress_fit))**2)

r2_nh = 1 - ss_res_nh / ss_tot
r2_mr = 1 - ss_res_mr / ss_tot

print(f'\nR-quadrado Neo-Hookean:   {r2_nh:.6f}')
print(f'R-quadrado Mooney-Rivlin: {r2_mr:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curvas completas
axes[0].plot(strain * 100, stress_exp, 'ko', markersize=1, label='Experimental', alpha=0.5)
axes[0].plot(strain[1:] * 100, neo_hookean(lam_fit, *popt_nh), 'r-',
             linewidth=2, label=f'Neo-Hookean (R2={r2_nh:.4f})')
axes[0].plot(strain[1:] * 100, mooney_rivlin(lam_fit, *popt_mr), 'b-',
             linewidth=2, label=f'Mooney-Rivlin (R2={r2_mr:.4f})')
axes[0].set_xlabel('Deformacao (%)')
axes[0].set_ylabel('Tensao (MPa)')
axes[0].set_title('Ajuste dos Modelos Constitutivos')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuos
res_nh = stress_fit - neo_hookean(lam_fit, *popt_nh)
res_mr = stress_fit - mooney_rivlin(lam_fit, *popt_mr)
axes[1].plot(strain[1:] * 100, res_nh, 'r-', alpha=0.7, label='Neo-Hookean')
axes[1].plot(strain[1:] * 100, res_mr, 'b-', alpha=0.7, label='Mooney-Rivlin')
axes[1].axhline(y=0, color='k', linestyle='--', linewidth=0.5)
axes[1].set_xlabel('Deformacao (%)')
axes[1].set_ylabel('Residuo (MPa)')
axes[1].set_title('Residuos dos Ajustes')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Comparacao com Literatura

Vamos apresentar os resultados ao Claude junto com o contexto do material
para comparacao com valores tipicos da literatura.


In [ ]:
lit_prompt = f"""Realizei a caracterizacao mecanica de um elastomero do tipo
borracha nitrílica (NBR) com 33% de acrilonitrila, vulcanizado com
enxofre e reforco de negro de fumo N550 (50 phr).

Resultados obtidos:
- Modulo inicial: {modulo_inicial:.2f} MPa
- M100: {M100:.2f} MPa
- M300: {M300:.2f} MPa
- Resistencia a tracao: {tensile_strength:.2f} MPa
- Alongamento na ruptura: {elongation_break:.0f}%
- Razao M300/M100: {M300/M100:.2f}

Modelos constitutivos ajustados:
- Neo-Hookean: C1={C1_nh:.4f} MPa (R2={r2_nh:.4f})
- Mooney-Rivlin: C1={C1_mr:.4f}, C2={C2_mr:.4f} MPa (R2={r2_mr:.4f})

Por favor:
1. Compare esses valores com faixas tipicas da literatura para NBR
   com composicao similar
2. Indique se os parametros C1 e C2 sao fisicamente razoaveis
3. Avalie a qualidade da vulcanizacao com base nos dados
4. Sugira melhorias na formulacao se necessario

Apresente em formato de tabela comparativa quando possivel."""

resp_lit = ask(lit_prompt, model=SONNET, max_tokens=2000)
print(resp_lit)

## Envelhecimento e Degradacao

Vamos gerar dados comparativos de amostras envelhecidas termicamente
e analisar os mecanismos de degradacao.


In [ ]:
# Gerar dados para amostra envelhecida (100C por 72h)
# Envelhecimento termico causa: aumento de rigidez, reducao de alongamento
np.random.seed(99)

strain_aged = np.linspace(0, 3.0, 300)  # Ruptura prematura em ~300%
lam_aged = 1 + strain_aged

C1_aged = 0.28   # Aumento de rigidez (reticulacao adicional)
C2_aged = 0.01   # Reducao da mobilidade molecular
C3_aged = 0.002  # Maior enrijecimento

stress_aged = 2 * (lam_aged - 1/lam_aged**2) * (C1_aged + C2_aged / lam_aged)
stress_aged += C3_aged * (lam_aged - 1)**3
stress_aged += np.random.normal(0, 0.03, len(strain_aged))
stress_aged[0] = 0
stress_aged = np.maximum(stress_aged, 0)

# Propriedades da amostra envelhecida
idx_100a = np.argmin(np.abs(strain_aged - 1.0))
M100_aged = stress_aged[idx_100a]
ts_aged = np.max(stress_aged)
eb_aged = strain_aged[np.argmax(stress_aged)] * 100

print('=== Comparacao Fresh vs Envelhecido ===')
print(f'{"Propriedade":<25} {"Fresh":>10} {"Envelhecido":>12} {"Variacao":>10}')
print('-' * 60)
print(f'{"M100 (MPa)":<25} {M100:>10.2f} {M100_aged:>12.2f} {(M100_aged/M100-1)*100:>+9.1f}%')
print(f'{"Resist. Tracao (MPa)":<25} {tensile_strength:>10.2f} {ts_aged:>12.2f} {(ts_aged/tensile_strength-1)*100:>+9.1f}%')
print(f'{"Along. Ruptura (%)":<25} {elongation_break:>10.0f} {eb_aged:>12.0f} {(eb_aged/elongation_break-1)*100:>+9.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(strain * 100, stress_exp, 'b-', linewidth=2, label='Amostra fresca')
ax.plot(strain_aged * 100, stress_aged, 'r-', linewidth=2, label='Envelhecida (100C, 72h)')

# Marcar M100
ax.plot(100, M100, 'bs', markersize=10)
ax.plot(100, M100_aged, 'rs', markersize=10)
ax.annotate(f'M100={M100:.2f}', (100, M100), textcoords='offset points',
            xytext=(10, 10), fontsize=9, color='blue')
ax.annotate(f'M100={M100_aged:.2f}', (100, M100_aged), textcoords='offset points',
            xytext=(10, -15), fontsize=9, color='red')

ax.set_xlabel('Deformacao (%)')
ax.set_ylabel('Tensao (MPa)')
ax.set_title('Efeito do Envelhecimento Termico no Elastomero')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
aging_prompt = f"""Realizei ensaios de tracao em um elastomero NBR (33% ACN,
50 phr N550, vulcanizado com enxofre) antes e apos envelhecimento
termico a 100 graus Celsius por 72 horas.

Resultados comparativos:

Amostra Fresca:
- M100: {M100:.2f} MPa
- Resistencia a tracao: {tensile_strength:.2f} MPa
- Alongamento na ruptura: {elongation_break:.0f}%

Amostra Envelhecida:
- M100: {M100_aged:.2f} MPa (+{(M100_aged/M100-1)*100:.1f}%)
- Resistencia a tracao: {ts_aged:.2f} MPa ({(ts_aged/tensile_strength-1)*100:+.1f}%)
- Alongamento na ruptura: {eb_aged:.0f}% ({(eb_aged/elongation_break-1)*100:+.1f}%)

Por favor analise:
1. Quais mecanismos de degradacao estao atuando?
2. A reticulacao oxidativa esta predominando sobre a cisao de cadeia?
3. O material atende normas automotivas tipicas (variacao max de 25% em M100)?
4. Sugira aditivos antioxidantes e antiozônio para melhorar a resistencia
5. Estime a vida util a 70 graus Celsius usando Arrhenius

Responda de forma tecnica e quantitativa quando possivel."""

resp_aging = ask(aging_prompt, model=SONNET, max_tokens=2000)
print(resp_aging)

## Laudo de Caracterizacao

Vamos pedir ao Claude para gerar um laudo tecnico completo com todos
os resultados obtidos.


In [ ]:
report_prompt = f"""Gere um laudo tecnico completo de caracterizacao de elastomero
com os seguintes dados:

MATERIAL: Borracha nitrilica (NBR) - 33% ACN
FORMULACAO: 50 phr negro de fumo N550, vulcanizacao com enxofre
NORMA: ASTM D412 (tracao), ASTM D573 (envelhecimento)

ENSAIO DE TRACAO (amostra fresca):
- Modulo inicial: {modulo_inicial:.2f} MPa
- M100: {M100:.2f} MPa
- M300: {M300:.2f} MPa
- Resistencia a tracao: {tensile_strength:.2f} MPa
- Alongamento na ruptura: {elongation_break:.0f}%
- Razao M300/M100: {M300/M100:.2f}

MODELOS CONSTITUTIVOS:
- Neo-Hookean: C1={C1_nh:.4f} MPa (R2={r2_nh:.4f})
- Mooney-Rivlin: C1={C1_mr:.4f}, C2={C2_mr:.4f} MPa (R2={r2_mr:.4f})

ENVELHECIMENTO (100C, 72h):
- M100: {M100_aged:.2f} MPa (variacao: {(M100_aged/M100-1)*100:+.1f}%)
- Resistencia: {ts_aged:.2f} MPa (variacao: {(ts_aged/tensile_strength-1)*100:+.1f}%)
- Alongamento: {eb_aged:.0f}% (variacao: {(eb_aged/elongation_break-1)*100:+.1f}%)

O laudo deve conter:
1. Objetivo
2. Metodologia
3. Resultados e Discussao
4. Comparacao com especificacoes
5. Conclusao e Recomendacoes

Use formato profissional de laboratorio de ensaios."""

report = ask(report_prompt, model=SONNET, max_tokens=3000)
print(report)

## Exercicios

1. **Modelo de Ogden**: Implemente o modelo de Ogden (3 parametros) e compare
   o ajuste com Neo-Hookean e Mooney-Rivlin. Qual modelo captura melhor o
   comportamento em altas deformacoes?

2. **Efeito Mullins**: Gere dados de carga-descarga ciclica com amolecimento
   progressivo (efeito Mullins). Peca ao Claude para explicar o fenomeno
   e sugerir um modelo constitutivo adequado.

3. **Variacao de formulacao**: Gere curvas para diferentes teores de negro de
   fumo (20, 40, 60 phr) e peca ao Claude para analisar a influencia do
   reforco nas propriedades mecanicas.

4. **Envelhecimento com ozonio**: Simule dados de degradacao por ozonio
   (formacao de trincas superficiais) e peca ao Claude para comparar os
   mecanismos com o envelhecimento termico.

5. **Analise estatistica**: Gere 5 repeticoes do ensaio com variabilidade
   experimental e calcule media, desvio padrao e intervalo de confianca
   para cada propriedade.


## Proximos Passos

- **Notebook 05**: Imageamento por microscopia eletronica - processamento
  de imagens e analise de falhas com Python e Claude
- Estudar modelos viscoelasticos para ensaios dinamico-mecanicos (DMA)
- Implementar analise de fadiga por ciclagem mecanica
- Explorar simulacao por elementos finitos com dados constitutivos

---
*Notebook gerado como material didatico do modulo 13_enfitec_projetos.*
